In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_contents, fetch_website_links
from IPython.display import Markdown, display, update_display

In [2]:
links = fetch_website_links("https://huggingface.co/")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/Qwen/Qwen3.6-35B-A3B',
 '/moonshotai/Kimi-K2.6',
 '/unsloth/Qwen3.6-35B-A3B-GGUF',
 '/tencent/HY-Embodied-0.5',
 '/baidu/ERNIE-Image',
 '/models',
 '/spaces/k2-fsa/OmniVoice',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview',
 '/spaces/webml-community/bonsai-webgpu',
 '/spaces/prithivMLmods/FireRed-Image-Edit-1.0-Fast',
 '/spaces/baidu/ERNIE-Image-Turbo',
 '/spaces',
 '/datasets/lambda/hermes-agent-reasoning-traces',
 '/datasets/Roman1111111/claude-opus-4.6-10000x',
 '/datasets/llamaindex/ParseBench',
 '/datasets/Jackrong/GLM-5.1-Reasoning-1M-Cleaned',
 '/datasets/Kassadin88/GLM-5.1-1000000x',
 '/datasets',
 '/join',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/inference/models',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google

In [3]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [5]:
print(get_links_user_prompt("https://huggingface.co/"))


Here is the list of links on the website https://huggingface.co/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/login
/join
/spaces
/models
/Qwen/Qwen3.6-35B-A3B
/moonshotai/Kimi-K2.6
/unsloth/Qwen3.6-35B-A3B-GGUF
/tencent/HY-Embodied-0.5
/baidu/ERNIE-Image
/models
/spaces/k2-fsa/OmniVoice
/spaces/r3gm/wan2-2-fp8da-aoti-preview
/spaces/webml-community/bonsai-webgpu
/spaces/prithivMLmods/FireRed-Image-Edit-1.0-Fast
/spaces/baidu/ERNIE-Image-Turbo
/spaces
/datasets/lambda/hermes-agent-reasoning-traces
/datasets/Roman1111111/claude-opus-4.6-10000x
/datasets/llamaindex/ParseBench
/datasets/Jackrong/GLM-5.1-Reasoning-1M-Cleaned
/datasets/Kassadin88/GLM-5.1-1000000x
/datasets
/join
/enterprise
/enterprise
/enterprise
/enterprise
/enterprise
/ent

In [7]:
load_dotenv(override=True)
base_url = os.getenv("OLLAMA_BASE_URL")
ollama = OpenAI(base_url=base_url, api_key="ollama")

In [8]:
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
suc

In [28]:
def select_relevant_links(url):
    print(f"Fetching relevant links for the website {url}")
    response = ollama.chat.completions.create(model="llama3.2", messages=[
        {"role": "system", "content": link_system_prompt},
        {'role': 'user', 'content': get_links_user_prompt(url)}
    ])
    result = response.choices[0].message.content.strip()
    # Strip markdown code fences if present
    if result.startswith("```"):
        result = result.split("```")[1]
        if result.startswith("json"):
            result = result[4:]
        result = result.strip()
    if not result:
        print("Model returned empty response, returning empty links")
        return {"links": []}
    try:
        data = json.loads(result)
        # print(f"Found {len(data['links'])} relevant links from {url}")
        return data
    except json.JSONDecodeError:
        print(f"Model returned non-JSON response: {result[:200]!r}")
        return {"links": []}

In [25]:
select_relevant_links("https://huggingface.co/")

Fetching relevant links for the website https://huggingface.co/
Model returned non-JSON response: 'Here are the relevant links for a brochure about Hugging Face:\n\n```\n{\n  "links": [\n    {"type": "About page", "url": "https://huggingface.co/"},\n    {"type": "Company page", "url": "https://blog.huggi'


{'links': []}

Generating the brochure from relevant links and website content

In [26]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [29]:
fetch_page_and_all_relevant_links("https://a24films.com/")

Fetching relevant links for the website https://a24films.com/


"## Landing Page:\n\nA24\n\nThe\nDrama\n2026\nMother\nMary\n2026\nBackrooms\n2026\nThe Death of Robin\nHood\n2026\nThe\nInvite\n2026\nThe Drama\n2026\nMother Mary\n2026\nBackrooms\n2026\nThe Death of Robin Hood\n2026\nThe Invite\n2026\n1\n5\nShop\nShop\nThe Drama: The Card Game\nShop Now\nPodcast\nPodcast\nGirl, Whatever with Charli xcx & Aidan Zamiri\nListen Now\nShop\nShop\nI Saw the TV Glow Screenplay Book\nShop Now\nWATCH NOW\nUndertone\nSHOP\nSHOP\nAAA24 Membership\nShop Now\nPODCAST\nPODCAST\nGreat American Dream Show with Josh Safdie & Sean Baker\nListen Now\nShop\nShop\nIf I Had Legs I'd Kick You Blu-ray\nShop Now\nWATCH NOW\nPillion\nPODCAST\nPODCAST\nHigh Octane with Rose Byrne & Emily Blunt\nListen Now\nSHOP\nSHOP\nBlack Logo Outline Crewneck\nShop Now\nSHOP\nSHOP\nMarc by Sofia\nShop Now\nWATCH NOW\nThe Moment\nShop\nShop\nThe Moment (The Score)\nShop Now\nSHOP\nSHOP\nMarty Supreme Blu-ray\nShop Now\nJobs\nShop\nApp\nMembership\nTerms of Use\nPrivacy Policy\nPrivacy Prefere

In [30]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [31]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [33]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(model="llama3.2", messages=[
        {'role': 'system', 'content': brochure_system_prompt},
        {'role': 'user', 'content': get_brochure_user_prompt(company_name, url)}
    ])
    result = response.choices[0].message.content
    display(Markdown(result))


In [35]:
create_brochure("Supabase", "https://supabase.com/")

Fetching relevant links for the website https://supabase.com/
Model returned non-JSON response: 'Here is the list of relevant links in JSON format:\n\n{\n    "links": [\n        {\n            "type": "home page",\n            "url": "https://supabase.com/"\n        },\n        {\n            "type": "Abo'


### Supabase: Empowering Innovation and Growth

At Supabase, we're on a mission to empower developers and startups to build and scale their own Postgres-based projects. Our platform provides a comprehensive set of tools to help you get started and succeed.

#### What We Offer

* **Full Postgres Database**: Start your project with a full-fledged Postgres database, trusted by fast-growing companies worldwide.
* **Authentication and Authorization**: Secure your data with Row Level Security (RLS) and built-in Auth.
* **Edge Functions**: Write custom code without deploying or scaling servers.
* **Storage and Realtime Integration**: Store and serve large files, and build multiplayer experiences with real-time data synchronization.
* **Vector Embeddings**: Integrate your favorite ML-models to store, index, and search vector embeddings.

#### Our Philosophy

We believe in giving you the freedom to work on your projects without unnecessary barriers. With Supabase, you can focus on building and innovating, knowing that our platform has got you covered.

### Customer Stories

* **Maergo's Express Delivery**: How Supaboxed Empowers Delivery Companies
Maergo achieved scalability, speed, and cost saving with Supabase. The company was able to scale from $0 to $1 million in 5 months.
* **Scaling Securely**: One Million Users Protected with Supabox Auth
The startup successfully protected one million users in just 7 months using Supabase's Auth features.

### Join Our Community

Ready to unleash your creativity and growth? Explore our resources, blog, and documentation to get started. Join our community of innovators and thought leaders to learn from each other and stay updated on the latest trends and insights.

### Careers at Supabase

We're always lookfor talented individuals to join our team. Check out our available positions and apply today!

---

Join the movement of entrepreneurs who choose innovation with Supabase. Let's build something amazing together!

In [36]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [37]:
stream_brochure("Elcarreira Technologies", "https://elcarreira.com/")

Fetching relevant links for the website https://elcarreira.com/
Model returned non-JSON response: 'Here is the list of relevant links in JSON format:\n\n\n{\n  "links": [\n    {\n      "type": "about page",\n      "url": "https://elcarreira.com/alumnos/temas-e-debates/que-sabe-alumno-sobre-empresas-sucess'


# Brochure: Elcarreira Technologies

**Empowering Intelligent Solutions**

Elcarreira Technologies is a leading developer of innovative software solutions, leveraging cutting-edge technologies to drive business success. Our mission is to become the trusted partner for organizations seeking intelligent solutions that enhance their operations and achieve their goals.

**Our Values**

At Elcarreira Technologies, we cultivate a culture of collaboration, innovation, and integrity. We believe in fostering an environment where our employees can grow professionally, contribute to meaningful projects, and feel valued as individuals. Our core values include:

*   Embracing change and innovation
*   Delivering exceptional results
*   Building strong relationships with clients and partners
*   Fostering a culture of transparency and open communication

**Customer Testimonials**

We serve a diverse range of customers across various industries, including finance, healthcare, retail, and manufacturing. Our clients trust us to provide customized solutions that address their unique challenges and goals.

*   "Elcarreira Technologies has been an invaluable partner in our digital transformation journey." - [Client Name], [Industry]
*   "Their team's expertise and commitment to excellence have helped us streamline operations and improve efficiency." - [Client Name], [Industry]

**Join Our Team**

Excited about joining a dynamic team of professionals who share your passion for innovation? Explore our current job openings at [Company Website]. We're looking for talented individuals with drive, creativity, and a desire to make a meaningful impact.

Ready to embark on a rewarding career journey with us? Browse our open positions today!

**Let's Connect**

Stay up-to-date with the latest news, insights, and company updates from Elcarreira Technologies. Follow us on social media or sign up for our newsletter to receive exclusive content and invitations to upcoming events.

[Company Social Media]
[Company Newsletter Sign-Up]